# Cluster & tier assignments

Runs `src.clustering.assign_tiers` on the forecast + PWC data to produce per-MSOA cluster labels and tiers for a chosen month.

**Inputs**
 `data/raw/centroids/msoa_2021_PWCs.csv` population-weighted centroids.
 `data/processed/wide_forecasts.csv` the `wide_forecasts` table built in `regression test.ipynb`.
  It carries `msoa_code`, `month`, and `intensity` columns, which is what `assign_tiers` needs.

Run Jupyter from the repo root so `import src` resolves.

In [45]:
import os
from pathlib import Path

# from dev manual: be at root so 'from src...' resolves.
if not Path("src").is_dir():
    os.chdir(Path.cwd().parent)

import pandas as pd

from src.config import DATA_DIR, RAW_DIR
from src.clustering import assign_tiers, load_pwc_coords

## 1. PWC coordinates
Raw columns `X, Y, MSOA21CD` renamed to `easting, northing, msoa_code`.

In [46]:
pwc = load_pwc_coords(pd.read_csv(RAW_DIR / "centroids" / "msoa_2021_PWCs.csv"))
pwc.head()

,msoa_code,easting,northing
0,E02000001,532290.3638,181745.9359
1,E02004372,582475.5406,110963.8205
2,E02006584,524353.1856,135416.2588
3,E02006411,504326.1721,170345.3420
4,E02003401,470327.0410,172513.7070


Prerequisite

In [47]:
# PREREQUISITE 
# do NOT run this cell here. It is commented out on purpose.                                    
                                                                                                      
# The next section reads data/processed/wide_forecasts.csv. That file is NOT                                
# in git (data/processed/ is gitignored), so it must be generated once from                                  
# the regression notebook before this notebook can run.                                             
                                                                                                            
# HOW TO GENERATE IT:                                                                                          
#   1. Open regression test.ipynb and run it top to bottom so the                                            
#      wide_forecasts DataFrame exists in that kernel's memory.                                              
#   2. With wide_forecasts still in memory, add a new cell at the bottom of                                  
#      regression test.ipynb containing exactly the code below, and run it.                                  
#   3. It writes the CSV to data/processed/. After that, this notebook can read                                
#      it and you do not need to rerun regression again unless forecasts change. 
#   4. after wide_forecasts.csv is saved, delete the code to avoid bloating the other notebook with non regression related code
                                                                                                             
#  Paste the following into regression test.ipynb, NOT into this notebook:                            
#                                                                                                              
# from src.config import DATA_DIR                                                                              
# DATA_DIR.mkdir(parents=True, exist_ok=True)                                                                  
# wide_forecasts.to_csv(DATA_DIR / "wide_forecasts.csv", index=False)                                          


## 2. Forecasts
`wide_forecasts` already contains `msoa_code`, `month`, `intensity`. `assign_tiers` slices the columns it needs

In [48]:
wide_forecasts = pd.read_csv(
    DATA_DIR / "wide_forecasts.csv", parse_dates=["month"]
)
wide_forecasts[["msoa_code", "month", "intensity"]].head()

,msoa_code,month,intensity
0,E02000001,2026-04-01,95897.337353
1,E02000001,2026-05-01,96968.920126
2,E02000001,2026-06-01,97840.604673
3,E02000001,2026-07-01,97630.016938
4,E02000001,2026-08-01,97634.355015


## 3. Pick a month
`assign_tiers` clusters one month at a time. Output available months:

In [49]:
months = sorted(wide_forecasts["month"].dt.strftime("%Y-%m-%d").unique())

In [50]:
months_threshold = {}
for MONTH in months:
    # Intensity cutoff for at-risk (TIER 2) noise points, taken from the specific month's distribution.
    month_intensity = wide_forecasts.loc[wide_forecasts["month"] == MONTH, "intensity"]
    threshold_value = 0.9
    THRESHOLD = month_intensity.quantile(threshold_value)
    months_threshold[MONTH] = THRESHOLD
    print(f"Threshold ({threshold_value}th pct of intensity for {MONTH}): {THRESHOLD:.3f}")

Threshold (0.9th pct of intensity for 2026-04-01): 15556.572
Threshold (0.9th pct of intensity for 2026-05-01): 15429.267
Threshold (0.9th pct of intensity for 2026-06-01): 15427.321
Threshold (0.9th pct of intensity for 2026-07-01): 15558.886
Threshold (0.9th pct of intensity for 2026-08-01): 15503.781
Threshold (0.9th pct of intensity for 2026-09-01): 15140.043
Threshold (0.9th pct of intensity for 2026-10-01): 15306.918
Threshold (0.9th pct of intensity for 2026-11-01): 15006.021
Threshold (0.9th pct of intensity for 2026-12-01): 14861.610
Threshold (0.9th pct of intensity for 2027-01-01): 14657.173
Threshold (0.9th pct of intensity for 2027-02-01): 14241.581
Threshold (0.9th pct of intensity for 2027-03-01): 14507.624


## 4. Cluster & assign tiers

In [ ]:
results = {}
for MONTH in months_threshold:
    result = assign_tiers(
    wide_forecasts,
    pwc,
    month=MONTH,
    threshold=months_threshold[MONTH],
    intensity_cap_quantile=0.95,
    min_cluster_size=5,
    )
    results[MONTH] = result



   msoa_code      easting     northing     intensity  cluster_label  \
0  E02000001  532290.3638  181745.9359  95897.337353            224   
1  E02000002  547988.0185  189412.9491   9689.155164             -1   
2  E02000003  548362.0317  188088.2498  15795.621326             -1   
3  E02000004  551020.1697  186865.3459   5584.097231             -1   
4  E02000005  548629.9425  186875.2643   9930.507955             -1   

   membership_prob    tier  final_weight  
0              1.0  Tier 1  95897.337353  
1              0.0  Tier 3    484.457758  
2              0.0  Tier 2   7897.810663  
3              0.0  Tier 3    279.204862  
4              0.0  Tier 3    496.525398  


## 5. Print results

In [56]:
for MONTH in results:
    print(results[MONTH].head())

   msoa_code      easting     northing     intensity  cluster_label  \
0  E02000001  532290.3638  181745.9359  95897.337353            224   
1  E02000002  547988.0185  189412.9491   9689.155164             -1   
2  E02000003  548362.0317  188088.2498  15795.621326             -1   
3  E02000004  551020.1697  186865.3459   5584.097231             -1   
4  E02000005  548629.9425  186875.2643   9930.507955             -1   

   membership_prob    tier  final_weight  
0              1.0  Tier 1  95897.337353  
1              0.0  Tier 3    484.457758  
2              0.0  Tier 2   7897.810663  
3              0.0  Tier 3    279.204862  
4              0.0  Tier 3    496.525398  
   msoa_code      easting     northing     intensity  cluster_label  \
0  E02000001  532290.3638  181745.9359  96968.920126            231   
1  E02000002  547988.0185  189412.9491   9348.095680            254   
2  E02000003  548362.0317  188088.2498  14598.665053             -1   
3  E02000004  551020.1697  1868

## 6. Save results in /data/processed

In [58]:
for MONTH in results:   
    (DATA_DIR / 'yearly_tiers').mkdir(parents=True, exist_ok=True)
    out_path = DATA_DIR / 'yearly_tiers' /f"tier_assignments_{MONTH}.csv"
    result.to_csv(out_path, index=False)
    print(out_path)

/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-04-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-05-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-06-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-07-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-08-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-09-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-10-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-11-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2026-12-01.csv
/home/rbilous/TUe-Multi-Disciplinary-CBL/data/processed/yearly_tiers/tier_assignments_2027-